# Intensywność emisyjna produkcji energii elektrycznej na świecie

**Projekt eksploracyjnej analizy danych (EDA)** przygotowany na podstawie pliku `owid-energy-data.csv`.

**Główna zmienna zależna:** `carbon_intensity_elec` — intensywność emisji CO₂ produkcji energii elektrycznej, mierzona w gramach CO₂ na kWh.

**Zakres głównej analizy:** przekrój krajów/terytoriów w **2022 r.**  
Wybrano 2022 r., ponieważ w tym pliku jest to najnowszy rok zapewniający bardzo wysokie pokrycie zmiennej zależnej i jednocześnie dostępność danych PKB. Dla porównania: 2024 ma tylko 86 niepustych obserwacji intensywności emisji i brak PKB, 2023 ma 172 obserwacje intensywności emisji i brak PKB, a 2022 ma 212 obserwacji intensywności emisji oraz 165 obserwacji PKB.

> **Uwaga metodologiczna:** analiza ma charakter eksploracyjny i przekrojowy. Zależności statystyczne nie są interpretowane jako związki przyczynowe.


## 1. Wprowadzenie

### Zbiór danych
Zbiór integruje wskaźniki dotyczące zużycia energii, miksu energetycznego, produkcji energii elektrycznej i innych zmiennych energetycznych.

W pliku jedna obserwacja odpowiada kombinacji **lokalizacja × rok**. Do głównej analizy przekrojowej usuwamy agregaty OWID (np. regiony i świat) i pozostawiamy jednostki z kodami ISO.

### Pytania badawcze
1. **Czy większy udział OZE w produkcji energii elektrycznej wiąże się z niższą intensywnością emisyjną?**
2. **Jak silnie udział paliw kopalnych wiąże się z intensywnością emisyjną?**
3. **Czy udział energetyki jądrowej, hydro, wiatru i fotowoltaiki różnicuje intensywność emisji?**
4. **Czy zamożność i skala zużycia energii na mieszkańca są wyraźnie związane z intensywnością emisyjną?**

Analizowane są więc relacje wielu zmiennych z jedną główną zmienną zależną, co odpowiada charakterowi analizy przekrojowej i eksploracyjnej.

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

DATA_PATH = Path("owid-energy-data.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/energy_eda_project/owid-energy-data.csv")

df = pd.read_csv(DATA_PATH)
print("Wymiary pełnego zbioru:", df.shape)
print("Zakres lat:", int(df["year"].min()), "-", int(df["year"].max()))
print("Liczba unikalnych lokalizacji:", df["country"].nunique())
df.head()

Wymiary pełnego zbioru: (23195, 130)
Zakres lat: 1900 - 2024
Liczba unikalnych lokalizacji: 314


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
3,ASEAN (Ember),2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
4,ASEAN (Ember),2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN


**Interpretacja.** Pełny plik ma **23 195 wierszy i 130 kolumn**  obejmuje lata **1900–2024** i zawiera **314 unikalnych nazw lokalizacji**. Ze względu na bardzo szeroki zakres czasowy i nierówną dostępność danych między latami dalsza analiza koncentruje się na porównywalnym przekroju 2022.